In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import plotly.graph_objects as go

In [ ]:
%cd './src'

import importlib
import ssgsea
import plot_ssgsea
import multimodal

importlib.reload(ssgsea)
importlib.reload(plot_ssgsea)
importlib.reload(multimodal)

%cd ..

In [ ]:
# load the preprocessed data
adata = ad.read_h5ad("BRCA_preprocessed.h5ad", backed=None)  
library_id='Visium_Human_Breast_Cancer'
adata

In [ ]:
# run the image segmentation for all values of sigma, save in a new anndata object, if not already done
adata_img = multimodal.run_img_segmentation(
                adata = adata,
                output_dir = "segmentation_results",
                sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
                superpixel = "slic",
                res = "hires"
                )

In [ ]:
# load the anndata object with the image segmentation results
adata_img = ad.read_h5ad("segmentation_adata/adata_img_segmentation_slic.h5ad")

In [ ]:
# run ssGSEA, if not done already
ssgsea = ssgsea.ssGSEA()
results = ssgsea.compute_ssgsea(adata, 'h.all.v2026.1.Hs.json', layer=None)
ssgsea_df = pd.DataFrame.from_dict(results)

In [ ]:
# load precomputed ssGSEA results
ssgsea_df = pd.read_csv('NES.csv', index_col=0)

In [ ]:
# run multimodal pipeline for EMT gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']],
    modality_combinations = [False, True],
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1)],  # no weighting
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_EMT"
    )

In [ ]:
# plot clustering outputs for EMT

results_adata_path = Path('multimodal_results_EMT/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    plot_ssgsea.show_results(ssgsea_df[['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']], 
                              gene_set='HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 
                              results_adata_dir=inp, output_dir=out)

In [ ]:
# run multimodal pipeline for ER late response gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_ESTROGEN_RESPONSE_LATE']],
    modality_combinations = [False, True], # how I want to combine gexpr and gsets
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_ER_late"
    )

In [ ]:
# plot clustering outputs for ER late response

results_adata_path = Path('multimodal_results_ER_late/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    plot_ssgsea.show_results(ssgsea_df[['HALLMARK_ESTROGEN_RESPONSE_LATE']], 
                              gene_set='HALLMARK_ESTROGEN_RESPONSE_LATE', 
                              results_adata_dir=inp, output_dir=out)

In [ ]:
# run multimodal pipeline for MYC targets V1 gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_MYC_TARGETS_V1']],
    modality_combinations = [False, True], # how I want to combine gexpr and gsets
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_MYC_V1"
    )

In [ ]:
# plot clustering outputs for MYC targets V1

results_adata_path = Path('multimodal_results_MYC_V1/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    plot_ssgsea.show_results(ssgsea_df[['HALLMARK_MYC_TARGETS_V1']], 
                              gene_set='HALLMARK_MYC_TARGETS_V1', 
                              results_adata_dir=inp, output_dir=out)